Use this code after you have generated the centroids based on the entire dataset using the notebook: CleanCalculateCentroids

Read in the packages that you will need.

In [1]:
#Import Packages
import pandas as pd
import seaborn as sns
import numpy as np
from scipy.stats import norm

import time
import sys
import matplotlib.pyplot as plt
import math
import os

from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import MiniBatchKMeans
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

Read in your raw annotated file. This must contain x,y spatial coordinates, cell type labels and neighborhood labels at minimum.

I will use a human intestine dataset from Hickey et al published in Nature in 2023.

In [ ]:
df = pd.read_csv(r"C:\Users\k1van\Box\PanOrgan\Data\Data_Downloaded\labeled_quantified\2023_Hickey_Nature_01\05_25_HuBMAP_tunit.csv")

In [ ]:
df

In [7]:
# for categorical data
def spatial_relplot(
    df,
    hue,
    region,
    size,
    x='x',
    y='y',
    region_col='unique_region',
    palette='bright',
    continuous_cmap='viridis',
    colorbar=False,
    title=None,
    invert_y=True,
    remove_axes=True,
    show=True,
):
    """
    Creates a spatial scatter plot using seaborn.relplot with clean scaling and styling.

    Parameters:
    - df: pandas DataFrame containing the data.
    - hue: column name for coloring (categorical or continuous).
    - region: the region name to filter from `region_col`.
    - x, y: column names for spatial coordinates.
    - region_col: column indicating regions (default: 'unique_region').
    - size: marker size.
    - palette: seaborn palette for categorical hue.
    - continuous_cmap: colormap name for continuous hue.
    - colorbar: set True for continuous hue.
    - title: plot title.
    - invert_y: flip y-axis (True by default).
    - remove_axes: remove axis ticks and labels (True by default).
    - show: whether to call plt.show().
    """
    data = df[df[region_col] == region].copy()

    # Invert y if needed
    if invert_y:
        data[y] = -data[y]

    # Determine if hue is continuous or categorical
    is_continuous = data[hue].dtype.kind in 'fc'

    # Set up the relplot
    g = sns.relplot(
        data=data,
        x=x,
        y=y,
        hue=hue,
        palette=palette if not is_continuous else None,
        height=8,
        aspect=1,
        kind='scatter',
        s=size,
        legend='full',
    )

    # Apply continuous colorbar if applicable
    if is_continuous:
        points = g.ax.collections[0]
        points.set_cmap(continuous_cmap)
        if colorbar:
            g.fig.colorbar(points, ax=g.ax, label=hue)

    # Style adjustments
    if remove_axes:
        g.set(xlabel=None, ylabel=None, xticks=[], yticks=[])
        sns.despine(left=True, bottom=True)

    if title:
        g.ax.set_title(title)

    if show:
        plt.show()

    return g


Set up the df to be called cells for neighborhood analysis.

In [ ]:
cells = df

Read in the centroids that were calculated in CleanCalculateCentroids.ipynb

In [4]:
df_centroids = pd.read_csv('/home/john-hickey/Desktop/Lab Member Project/Kyra_Van_Batavia/In-house/PanOrgan/SO_Project2/all_neighborhood_centroids')

In [ ]:
df_centroids

Run Neighborhood Analysis: Now, we will perform neighborhood analysis on the data to get the nearest neighbor vectors for all cells in the dataset. We will use defined functions to split the image into windows and specify the number of k nearest neighbors that we would like to use for our analysis.

In [6]:
#Function for getting neighborhood windows
def get_windows(job, n_neighbors):

    # Unpack the job tuple containing start_time, idx, tissue_name, and indices
    start_time, idx, tissue_name, indices = job

    # Record the current time to measure the duration of the job
    job_start = time.time()

    # Print a message indicating the start of the job
    print("Starting:", str(idx+1)+'/'+str(len(exps)), ': ' + exps[idx])

    # Get the subset of the dataset for the specific tissue
    tissue = tissue_group.get_group(tissue_name)

    # Extract the coordinates (X, Y) for the points to be fitted from the tissue subset
    to_fit = tissue.loc[indices][[X, Y]].values

    # Fit the NearestNeighbors model on the tissue's X, Y coordinates
    fit = NearestNeighbors(n_neighbors=n_neighbors).fit(tissue[[X, Y]].values)

    # Find the nearest neighbors for the points in 'to_fit'
    m = fit.kneighbors(to_fit)

    # Sort the neighbors
    # 'args' are the indices that would sort the distances
    args = m[0].argsort(axis=1)

    # 'add' is used to adjust indices for flattened array
    add = np.arange(m[1].shape[0]) * m[1].shape[1]

    # Calculate sorted indices for neighbors
    sorted_indices = m[1].flatten()[args + add[:, None]]

    # Retrieve the neighbor indices from the tissue dataset
    neighbors = tissue.index.values[sorted_indices]

    # Record the end time of the job
    end_time = time.time()

    # Print a message indicating the end of the job and the duration
    print("Finishing:", str(idx+1)+"/"+str(len(exps)), ": "+ exps[idx], end_time - job_start, end_time - start_time)

    # Return the neighbor indices as an array of integers
    return neighbors.astype(np.int32)

The next block of code will help the code find cellhier to run neighborhood analysis.

In [ ]:
cellhier_path = '/home/john-hickey/Desktop/Lab Member Project/Kyra_Van_Batavia/In-house/PanOrgan/cellhier'
sys.path.append('/home/john-hickey/Desktop/Lab Member Project/Kyra_Van_Batavia/In-house/PanOrgan/cellhier')
from general import *
from plot_john import *

Reset the index of the dataframe for neighborhood analysis to function properly.

In [ ]:
cells.reset_index(inplace=True, drop=True)
print(cells)

In [10]:
# Define column names that will be used for neighborhood analysis
X = 'x'                  # Variable for the X coordinate
Y = 'y'                  # Variable for the Y coordinate
reg = 'unique_region'         # Variable for the filename or region identifier associated with coordinates
cluster_col = 'Cell Type'  # Variable for cell type/subtype classification

# List of columns to keep for analysis
keep_cols = [X, Y, reg, cluster_col]

In [11]:
# Concatenate the original 'cells' DataFrame with dummy variables created from 'cluster_col'
# pd.get_dummies() converts categorical variable(s) into dummy/indicator variables
cells = pd.concat([cells, pd.get_dummies(cells[cluster_col])], axis=1)

# Get unique values from the 'cluster_col' column to use for summarization
sum_cols = cells[cluster_col].unique()

# Retrieve the values for these unique categories as a NumPy array
# This array can be used for further analysis or operations later for calculating the neighborhoods
values = cells[sum_cols].values

In [12]:
#We can choose a range of nearest neighbors to calculate the neighborhoods
ks = [5,10,20] # k=5 means it collects 5 nearest neighbors for each center cell
n_neighbors = max(ks) #sets n_neighbors to max of the list that is set

In [ ]:
# Group the cell data by region
# 'cells' is a DataFrame containing cell data
# 'tissue_group' will be a GroupBy object with cells grouped by the 'reg' column (representing regions)
tissue_group = cells[[X, Y, reg]].groupby(reg)

# Get a list of unique regions (filenames)
# 'exps' will contain all unique region names found in the 'reg' column of the 'cells' DataFrame
exps = list(cells[reg].unique())

# Prepare chunks of data for processing
# 'tissue_chunks' is a list of tuples, each tuple representing a job for processing
# Each tuple contains the current time, index of the region in 'exps', the region name, and a subset of indices
# 'np.array_split(indices, 1)' splits the indices for each group into chunks (1 chunk in this case)
# This loop goes through each group in 'tissue_group', and for each group, it creates a job tuple
tissue_chunks = [(time.time(), exps.index(t), t, a) for t, indices in tissue_group.groups.items() for a in np.array_split(indices, 1)]

# Process each job to get the windows (neighbors of the cells)
# 'tissues' is a list of results from the 'get_windows' function
# The 'get_windows' function is applied to each job in 'tissue_chunks'
# 'n_neighbors' is a parameter for the 'get_windows' function, defining the number of neighbors to consider
tissues = [get_windows(job, n_neighbors) for job in tissue_chunks]

In [14]:
# Initialize a dictionary to store the output
out_dict = {}

# Loop over a list of values 'ks' (different numbers of neighbors to consider)
for k in ks:
    # Iterate over each tissue's neighbors and the corresponding job information
    for neighbors, job in zip(tissues, tissue_chunks):

        # Create an array of indices for the current chunk of data
        chunk = np.arange(len(neighbors))  # equivalent to 0, 1, 2, ..., len(neighbors)-1

        # Extract the tissue name and indices from the job tuple
        tissue_name = job[2]  # Region/filename from the job tuple
        indices = job[3]      # Indices from the job tuple

        # Compute the 'window' - a summary measure for the neighborhood of each cell up to the k-th neighbor
        # Reshape and sum values to get a compact representation of neighborhood information
        window = values[neighbors[chunk, :k].flatten()].reshape(len(chunk), k, len(sum_cols)).sum(axis=1)

        # Store the computed window and indices in the output dictionary
        # Keyed by a tuple of (tissue_name, k)
        out_dict[(tissue_name, k)] = (window.astype(np.float16), indices)

# Initialize a dictionary to store the final windows data
windows = {}

# Iterate over each value of k again to process the stored information
for k in ks:

    # Concatenate data for each experiment ('exp') into a DataFrame
    # This DataFrame contains the window data for each cell, indexed by cell indices, for the current value of k
    window = pd.concat([pd.DataFrame(out_dict[(exp, k)][0], index=out_dict[(exp, k)][1].astype(int), columns=sum_cols) for exp in exps], axis=0)

    # Ensure the window data is in the same order as the original cells DataFrame
    window = window.loc[cells.index.values]

    # Concatenate the window data with the original columns specified in 'keep_cols'
    window = pd.concat([cells[keep_cols], window], axis=1)

    # Store the concatenated DataFrame in the 'windows' dictionary, keyed by the current value of k
    windows[k] = window


In [15]:
#Choose k value to analyze and pull out from dictionary of stored results of vectors
k = 10
windows2 = windows[k]
#Add cell type column to output windows dataframe
windows2[cluster_col] = cells[cluster_col]

In [ ]:
windows2

Run Mixture Model to Find Probabilities

Importantly in the next step below, we assume that when the standard deviation value for the centroid is zero, if the cell type vector value for a cell is equal to the mean then probability is 1. This means that if the value is not equal to the mean, the probability is 0.

We will use the dataframe of nearest neighbor vectors that we just computed in neighborhood analysis and compare them to the annoated neighborhood centroid distributions from the mean and standard deviations that we calculated in CleanCalculateCentroids. This code will calculate the probability that each cell belongs in each neighborhood based on an assumed gaussian normal distribution for the neighbor cell type vectors.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from multiprocessing import Pool, cpu_count
from tqdm import tqdm

# List of neighborhoods to loop through, update this for other datasets
neighborhoods_to_loop = [
    'Innate Immune Enriched', 'Outer Follicle', 'Plasma Cell Enriched',
    'Transit Amplifying Zone', 'Adaptive Immune Enriched', 'Stroma',
    'Paneth Enriched', 'Smooth Muscle & Innate Immune', 'Mature Epithelial',
    'Microvasculature', 'CD8+ T Enriched IEL', 'Stroma & Innate Immune',
    'Macrovasculature', 'Innervated Stroma', 'Secretory Epithelial',
    'Innervated Smooth Muscle', 'Smooth Muscle', 'Glandular Epithelial',
    'CD66+ Mature Epithelial', 'Inner Follicle'
]

# List of cell types, update this for other datasets
cell_type_features = [
    'NK', 'Enterocyte', 'MUC1+ Enterocyte', 'TA', 'CD66+ Enterocyte', 'Paneth', 'Goblet',
    'Neuroendocrine', 'CD57+ Enterocyte', 'Smooth muscle', 'Lymphatic',
    'CD8+ T', 'DC', 'M2 Macrophage', 'M1 Macrophage', 'B', 'Neutrophil',
    'Endothelial', 'Cycling TA', 'Plasma', 'CD4+ T cell', 'Stroma', 'Nerve',
    'ICC', 'CD7+ Immune'
]

# Function to calculate probabilities for a single cell
def calculate_probabilities_for_cell(args):
    cell_data, df_centroids, cell_type_features = args  # Unpack the arguments

    neighborhood_probs = {}

    # Loop through each neighborhood (each row in df_centroids)
    for _, centroid_row in df_centroids.iterrows():
        neighborhood_name = centroid_row['Neighborhood']
        total_prob = 1

        # For each cell type in the list, calculate the probability
        for cell_type in cell_type_features:
            mean_col = f'{cell_type}_mean'
            std_col = f'{cell_type}_std'

            if mean_col in centroid_row and std_col in centroid_row:
                mean = centroid_row[mean_col]
                std = centroid_row[std_col]

                # Get the value of the current cell for this cell type (nearest neighbor count)
                cell_value = cell_data.get(cell_type, np.nan)

                # If std is zero, check if the cell value matches the mean
                if std == 0:
                    if cell_value == mean:
                        cell_prob = 1
                    else:
                        cell_prob = 0
                else:
                    # Otherwise, calculate the probability using the normal distribution (PDF)
                    cell_prob = norm.pdf(cell_value, loc=mean, scale=std)

                # Multiply the probability for this cell type to the total probability
                total_prob *= cell_prob

        # Store the total probability for the current neighborhood
        neighborhood_probs[neighborhood_name] = total_prob

    # Normalize the probabilities across all neighborhoods so they sum to 1
    total_prob_sum = sum(neighborhood_probs.values())
    for neighborhood in neighborhood_probs:
        neighborhood_probs[neighborhood] /= total_prob_sum

    return neighborhood_probs

# Function to parallelize the calculations across all cells
def parallelize_probability_calculations(windows2, df_centroids, cell_type_features):
    # Use all available CPUs for parallel processing
    num_processes = cpu_count()

    # Create a pool of workers
    with Pool(num_processes) as pool:
        # Use tqdm to display progress bar
        results = list(tqdm(pool.imap(
            calculate_probabilities_for_cell,  # Pass the function directly
            [(windows2.loc[cell_index], df_centroids, cell_type_features) for cell_index in windows2.index]), total=len(windows2)))

    return results

# Call the parallelization function
probabilities_list = parallelize_probability_calculations(windows2, df_centroids, cell_type_features)

# Convert the results into a DataFrame
probabilities_df = pd.DataFrame(probabilities_list, index=windows2.index)

# Print or save the resulting probabilities DataFrame
print(probabilities_df)

probabilities_df contains the information for all the cells and each of their probabilities for each of the possible neighborhoods in the dataset. We can investigate it to make sure there are no NaN values and it looks as we assume.

In [ ]:
probabilities_df

In [19]:
# Save probabilities results as CSV
probabilities_df.to_csv('/home/john-hickey/Desktop/Lab Member Project/Kyra_Van_Batavia/In-house/PanOrgan/SO_Project2/all_cells_all_neighborhood_probs', index=True)

In [ ]:
# Merge the two dataframes on their index
merged_df = probabilities_df.join(df)

merged_df

In [ ]:
# Save the merged dataframe as a CSV file
merged_df.to_csv('intestine_all_information_2.csv')

print("/home/john-hickey/Desktop/Lab Member Project/Kyra_Van_Batavia/In-house/PanOrgan/SO_Project2/Merged dataframe has been saved as 'intestine_all_information_2.csv'")

Now, we will plot our results. Read in the .csv file containing the probability information for all cells in the dataset for every possible neighborhood.

In [3]:
probabilities_df = pd.read_csv(r'C:\Users\k1van\Box\Home Folder kv120\Private\Research\PanOrgan\SO_Project2\all_cells_all_neighborhood_probs')

This code will determine the cell's assigned neighborhood probability by matching the index in the original raw data dataframe (df) to the probabilities dataframe (probabilities_df). Then, the code will plot the raw probabilities for each of the cell's assigned neighborhood in spatial coordinates, separating panels for each unique region in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Step 1: Retrieve assigned neighborhoods from df
assigned_neighborhoods = df['Neighborhood']

# Step 2: Extract the probability corresponding to each cell's assigned neighborhood from probabilities_df
assigned_probabilities = probabilities_df.reindex(df.index).apply(
    lambda row: row[df.loc[row.name, 'Neighborhood']], axis=1
)

# Step 3: Create a new DataFrame to store x, y, assigned neighborhood, probability, and region
visualization_df = pd.DataFrame({
    'x': df['x'],
    'y': df['y'],
    'assigned_neighborhood': assigned_neighborhoods,
    'assigned_probability': assigned_probabilities,
    'unique_region': df['unique_region']  # Adding region for FacetGrid
})

# Step 4: Create a FacetGrid to separate the data by 'unique_region'
g = sns.FacetGrid(visualization_df, col="unique_region", col_wrap=4, height=5)  # Adjust col_wrap for the number of columns

# Map scatter plot without passing 'c' explicitly to avoid the conflict
g.map(plt.scatter, 'x', 'y', alpha=0.75, s=5)

# Add the color for the scatter plot manually by setting color based on 'assigned_probability'
for ax in g.axes.flat:
    scatter = ax.collections[0]  # Get the scatter plot collections
    scatter.set_facecolor(plt.cm.plasma(visualization_df['assigned_probability'].values))  # Apply color map

# Add colorbar
g.fig.colorbar(scatter, label="Assigned Neighborhood Probability")

# Adjust the labels and titles
g.set_axis_labels('X Position', 'Y Position')
g.set_titles('Region: {col_name}')
g.fig.suptitle('Cell Positions Colored by Assigned Probability Using All Neighborhood Centroids', fontsize=16)
g.tight_layout()
g.fig.subplots_adjust(top=0.9)  # Adjust to make room for the suptitle

# Show the plot
plt.show()


For plotting only cells in one neighborhood.

In [ ]:
filtered_cells = df[df['Neighborhood']== 'Outer Follicle']

filtered_probabilities_df = probabilities_df.loc[filtered_cells.index]

filtered_cells

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Step 1: Retrieve assigned neighborhoods
assigned_neighborhoods = filtered_cells['Neighborhood']

# Step 2: Extract the probability corresponding to each cell's assigned neighborhood
assigned_probabilities = filtered_probabilities_df.reindex(filtered_cells.index).apply(
    lambda row: row[filtered_cells.loc[row.name, 'Neighborhood']], axis=1
)

# Step 3: Create a new DataFrame to store x, y, assigned neighborhood, probability, and region
visualization_df = pd.DataFrame({
    'x': filtered_cells['x'],  # Use filtered_cells for x and y coordinates
    'y': filtered_cells['y'],
    'assigned_neighborhood': assigned_neighborhoods,
    'assigned_probability': assigned_probabilities,
    'unique_region': filtered_cells['unique_region']  # Use filtered_cells for the region
})

# Step 4: Create a FacetGrid to separate the data by 'unique_region'
g = sns.FacetGrid(visualization_df, col="unique_region", col_wrap=4, height=5)  # Adjust col_wrap for the number of columns

# Map scatter plot and color points by assigned probability
# Specify 'c' for color and apply the 'viridis' colormap
g.map(plt.scatter, 'x', 'y', alpha=0.75, s=1)

# Add color manually
for ax in g.axes.flat:
    # Extract the assigned probability values for each plot
    scatter = ax.collections[0]  # Get the scatter plot collections
    scatter.set_facecolor(plt.cm.viridis(visualization_df['assigned_probability'].values))  # Apply color map

# Add colorbar
g.fig.colorbar(g.axes[0].collections[0], label="Assigned Neighborhood Probability")

# Adjust the labels and titles
g.set_axis_labels('X Position', 'Y Position')
g.set_titles('Region: {col_name}')
g.fig.suptitle('Cell Positions Colored by Assigned Probability Using All Neighborhood Centroids', fontsize=16)
g.tight_layout()
g.fig.subplots_adjust(top=0.9)  # Adjust to make room for the suptitle

# Show the plot
plt.show()


Follicle Structures: B009_Trans, B008_Sigmoid, B006_Decending - Sigmoid, B009_Ilium, B004_Ascending, B005_Proximal Jejunum

Now, we will plot a few of the regions where we see follicle neighborhoods above. 

Plot B006_Decending-Sigmoid

In [ ]:
# Filter the entire dataframe for the unique region that you want to plot
filtered_cells = df[df['unique_region']== 'B006_Descending - Sigmoid']

filtered_probabilities_df = probabilities_df.loc[filtered_cells.index]

filtered_cells


Define new cellhier_path with updated catplot2 function that allows to update the size of words in the plot (plot_john_2).

In [5]:
cellhier_path = r'C:\Users\k1van\Box\Home Folder kv120\Private\Research\PanOrgan\cellhier'
sys.path.append(r'C:\Users\k1van\Box\Home Folder kv120\Private\Research\PanOrgan\cellhier')
from general import *
from plot_john_2 import *

Plot the region's cells' assigned neighborhoods on spatial coordinates.

In [ ]:

# Step 1: Retrieve assigned neighborhoods and probabilities for the filtered cells
assigned_neighborhoods = filtered_cells['Neighborhood']
assigned_probabilities = filtered_probabilities_df.reindex(filtered_cells.index).apply(
    lambda row: row[filtered_cells.loc[row.name, 'Neighborhood']], axis=1
)

# Step 2: Create a new DataFrame for plotting
visualization_df = pd.DataFrame({
    'x': filtered_cells['x'],
    'y': filtered_cells['y'],
    'Assigned Neighborhood': assigned_neighborhoods,
    'Assigned Probability': assigned_probabilities,
    'unique_region': filtered_cells['unique_region']
})

# Step 3: Filter for region of interest
desired_region = 'B006_Descending - Sigmoid'

#modify figure size aesthetics for each neighborhood
plt.rcParams["legend.markerscale"] = 15
figs = catplot2(visualization_df, X = 'x', Y='y', exp = 'unique_region',
               hue = 'Assigned Neighborhood', axis='off', invert_y=False, 
               size = 1, figsize=10, legend=True, legend_title_fontsize=35, legend_fontsize=35, title_fontsize=35, palette='tab20',dpi=300)

Plot the region's cells' assigned probabilities on spatial coordinates.

In [ ]:
# Step 1: Retrieve assigned neighborhoods and probabilities for the filtered cells
assigned_neighborhoods = filtered_cells['Neighborhood']

assigned_probabilities = filtered_probabilities_df.reindex(filtered_cells.index).apply(
    lambda row: row[filtered_cells.loc[row.name, 'Neighborhood']], axis=1
)

# Step 2: Create a new DataFrame with x, y, assigned neighborhood, probability, and region
visualization_df = pd.DataFrame({
    'x': filtered_cells['x'],  # Use filtered_cells for x and y coordinates
    'y': filtered_cells['y'],
    'assigned_neighborhood': assigned_neighborhoods,
    'assigned_probability': assigned_probabilities,
    'unique_region': filtered_cells['unique_region']  # Use filtered_cells for the region
})

# Step 3: Filter for the desired unique_region (e.g., 'B006_Descending - Sigmoid')
desired_region = 'B006_Descending - Sigmoid'  # You can replace this with any other region you'd like to filter by
filtered_region_df = visualization_df[visualization_df['unique_region'] == desired_region]

# Step 4: Create the first plot (Assigned Probability)
plt.figure(figsize=(10, 10))  # Set the figure size
sc1 = plt.scatter(
    filtered_region_df['x'], filtered_region_df['y'],
    c=filtered_region_df['assigned_probability'], cmap='viridis', alpha=0.75, s=6
)

# Title with larger font
title = f'Assigned Probability\n({desired_region})'
plt.title(title, fontsize=35)

# Add colorbar and customize font size
cbar = plt.colorbar(sc1)
cbar.set_label("Assigned Probability", fontsize=30)
cbar.ax.tick_params(labelsize=30)  # Set tick label size

# Remove axis and axis labels
plt.axis('off')

# Ensure the figure fits the data without squishing
plt.tight_layout()

# Show the first plot
plt.show()



B009_Trans

In [ ]:
filtered_cells = df[df['unique_region']== 'B009_Trans']

filtered_probabilities_df = probabilities_df.loc[filtered_cells.index]

filtered_cells

Plotting for generating figures for paper, zoomed in on the follicle.

In [ ]:
import matplotlib.pyplot as plt

# Step 1: Retrieve assigned neighborhoods and probabilities for the filtered cells
assigned_neighborhoods = filtered_cells['Neighborhood']

assigned_probabilities = filtered_probabilities_df.reindex(filtered_cells.index).apply(
    lambda row: row[filtered_cells.loc[row.name, 'Neighborhood']], axis=1
)

# Step 2: Create a new DataFrame with x, y, assigned neighborhood, probability, and region
visualization_df = pd.DataFrame({
    'x': filtered_cells['x'],
    'y': filtered_cells['y'],
    'assigned_neighborhood': assigned_neighborhoods,
    'assigned_probability': assigned_probabilities,
    'unique_region': filtered_cells['unique_region']
})

# Step 3: Filter for the desired unique_region
desired_region = 'B009_Trans'
filtered_region_df = visualization_df[visualization_df['unique_region'] == desired_region]

# Set zoom limits (update these values to zoom as needed)
xlim_coords = (7000, 9750)  # Example: zoom in on x from 1000 to 2000
ylim_coords = (0, 3000)  # Example: zoom in on y from 1500 to 2500

# Step 4: Create the scatter plot without a colorbar
plt.figure(figsize=(10, 10), dpi=300)
plt.scatter(
    filtered_region_df['x'], filtered_region_df['y'],
    c=filtered_region_df['assigned_probability'], cmap='viridis', alpha=0.75, s=6
)

# Set zoom limits
plt.xlim(*xlim_coords)
plt.ylim(*ylim_coords)

# Add title
plt.title(f'Assigned Probability\n({desired_region})', fontsize=35)

# Hide axes
plt.axis('off')

# Adjust layout
plt.tight_layout()

# Show the plot
plt.show()


Plotting for investigating the neighborhoods and probability alignment.

In [ ]:
# Step 1: Retrieve assigned neighborhoods and probabilities for the filtered cells
assigned_neighborhoods = filtered_cells['Neighborhood']

assigned_probabilities = filtered_probabilities_df.reindex(filtered_cells.index).apply(
    lambda row: row[filtered_cells.loc[row.name, 'Neighborhood']], axis=1
)

# Step 2: Create a new DataFrame with x, y, assigned neighborhood, probability, and region
visualization_df = pd.DataFrame({
    'x': filtered_cells['x'],  # Use filtered_cells for x and y coordinates
    'y': filtered_cells['y'],
    'assigned_neighborhood': assigned_neighborhoods,
    'assigned_probability': assigned_probabilities,
    'unique_region': filtered_cells['unique_region']  # Use filtered_cells for the region
})

# Step 3: Filter for the desired unique_region (e.g., 'B006_Descending - Sigmoid')
desired_region = 'B009_Trans'  # You can replace this with any other region you'd like to filter by
filtered_region_df = visualization_df[visualization_df['unique_region'] == desired_region]

# Step 4: Create the first plot (Assigned Probability)
plt.figure(figsize=(12, 12), dpi=600)
sc1 = plt.scatter(
    filtered_region_df['x'], filtered_region_df['y'],
    c=filtered_region_df['assigned_probability'], cmap='viridis', alpha=0.75, s=5
)
plt.xlabel('X Position')
plt.ylabel('Y Position')
plt.title(f'Assigned Probability ({desired_region})')
plt.colorbar(sc1, label="Assigned Probability")
plt.tight_layout()
plt.show()

# Step 5: Create the second plot (Assigned Neighborhood with unique colors)
# Ensure each neighborhood gets a unique color
neighborhood_palette = sns.color_palette("tab20", len(filtered_region_df['assigned_neighborhood'].unique()))

# Map neighborhoods to unique colors
neighborhood_color_map = filtered_region_df['assigned_neighborhood'].map(
    lambda x: neighborhood_palette[list(filtered_region_df['assigned_neighborhood'].unique()).index(x)]
)

plt.figure(figsize=(12, 12), dpi=600)
sc2 = plt.scatter(
    filtered_region_df['x'], filtered_region_df['y'],
    c=neighborhood_color_map, alpha=0.75, s=5
)

plt.xlabel('X Position')
plt.ylabel('Y Position')
plt.title(f'Assigned Neighborhood ({desired_region})')

# Custom legend with neighborhood names
handles = []
for i, neighborhood in enumerate(filtered_region_df['assigned_neighborhood'].unique()):
    handle = plt.Line2D([0], [0], marker='o', color='w', label=neighborhood,
                        markersize=10, markerfacecolor=neighborhood_palette[i])
    handles.append(handle)

# Add the legend outside the plot
plt.legend(handles=handles, title="Neighborhood", bbox_to_anchor=(1.05, 1), loc='upper left')

# Adjust layout to avoid overlap
plt.tight_layout()

# Show the second plot
plt.show()


B004_Ascending

In [ ]:
filtered_cells = df[df['unique_region']== 'B004_Ascending']

filtered_probabilities_df = probabilities_df.loc[filtered_cells.index]

filtered_cells

Generate the figure for the paper which is zoomed in on the follicle.

In [ ]:
import matplotlib.pyplot as plt

# Step 1: Retrieve assigned neighborhoods and probabilities for the filtered cells
assigned_neighborhoods = filtered_cells['Neighborhood']

assigned_probabilities = filtered_probabilities_df.reindex(filtered_cells.index).apply(
    lambda row: row[filtered_cells.loc[row.name, 'Neighborhood']], axis=1
)

# Step 2: Create a new DataFrame with x, y, assigned neighborhood, probability, and region
visualization_df = pd.DataFrame({
    'x': filtered_cells['x'],
    'y': filtered_cells['y'],
    'assigned_neighborhood': assigned_neighborhoods,
    'assigned_probability': assigned_probabilities,
    'unique_region': filtered_cells['unique_region']
})

# Step 3: Filter for the desired unique_region
desired_region = 'B004_Ascending'
filtered_region_df = visualization_df[visualization_df['unique_region'] == desired_region]

# Set zoom limits (update these values to zoom as needed)
xlim_coords = (6000, 10000)  # Example: zoom in on x from 1000 to 2000
ylim_coords = (5000, 8000)  # Example: zoom in on y from 1500 to 2500

# Step 4: Create the scatter plot without a colorbar
plt.figure(figsize=(10, 10), dpi=300)
plt.scatter(
    filtered_region_df['x'], filtered_region_df['y'],
    c=filtered_region_df['assigned_probability'], cmap='viridis', alpha=0.75, s=6
)

# Set zoom limits
plt.xlim(*xlim_coords)
plt.ylim(*ylim_coords)

# Add title
plt.title(f'Assigned Probability\n({desired_region})', fontsize=35)

# Hide axes
plt.axis('off')

# Adjust layout
plt.tight_layout()

# Show the plot
plt.show()


Plotting for investigating the neighborhood and probabilities.

In [ ]:
# Step 1: Retrieve assigned neighborhoods and probabilities for the filtered cells
assigned_neighborhoods = filtered_cells['Neighborhood']

assigned_probabilities = filtered_probabilities_df.reindex(filtered_cells.index).apply(
    lambda row: row[filtered_cells.loc[row.name, 'Neighborhood']], axis=1
)

# Step 2: Create a new DataFrame with x, y, assigned neighborhood, probability, and region
visualization_df = pd.DataFrame({
    'x': filtered_cells['x'],  # Use filtered_cells for x and y coordinates
    'y': filtered_cells['y'],
    'assigned_neighborhood': assigned_neighborhoods,
    'assigned_probability': assigned_probabilities,
    'unique_region': filtered_cells['unique_region']  # Use filtered_cells for the region
})

# Step 3: Filter for the desired unique_region (e.g., 'B006_Descending - Sigmoid')
desired_region = 'B004_Ascending'  # You can replace this with any other region you'd like to filter by
filtered_region_df = visualization_df[visualization_df['unique_region'] == desired_region]

# Step 4: Create the first plot (Assigned Probability)
plt.figure(figsize=(12, 12), dpi=600)
sc1 = plt.scatter(
    filtered_region_df['x'], filtered_region_df['y'],
    c=filtered_region_df['assigned_probability'], cmap='viridis', alpha=0.75, s=5
)
plt.xlabel('X Position')
plt.ylabel('Y Position')
plt.title(f'Assigned Probability ({desired_region})')
plt.colorbar(sc1, label="Assigned Probability")
plt.tight_layout()
plt.show()

# Step 5: Create the second plot (Assigned Neighborhood with unique colors)
# Ensure each neighborhood gets a unique color
neighborhood_palette = sns.color_palette("tab20", len(filtered_region_df['assigned_neighborhood'].unique()))

# Map neighborhoods to unique colors
neighborhood_color_map = filtered_region_df['assigned_neighborhood'].map(
    lambda x: neighborhood_palette[list(filtered_region_df['assigned_neighborhood'].unique()).index(x)]
)

plt.figure(figsize=(12, 12), dpi=600)
sc2 = plt.scatter(
    filtered_region_df['x'], filtered_region_df['y'],
    c=neighborhood_color_map, alpha=0.75, s=5
)

plt.xlabel('X Position')
plt.ylabel('Y Position')
plt.title(f'Assigned Neighborhood ({desired_region})')

# Custom legend with neighborhood names
handles = []
for i, neighborhood in enumerate(filtered_region_df['assigned_neighborhood'].unique()):
    handle = plt.Line2D([0], [0], marker='o', color='w', label=neighborhood,
                        markersize=10, markerfacecolor=neighborhood_palette[i])
    handles.append(handle)

# Add the legend outside the plot
plt.legend(handles=handles, title="Neighborhood", bbox_to_anchor=(1.05, 1), loc='upper left')

# Adjust layout to avoid overlap
plt.tight_layout()

# Show the second plot
plt.show()


B005_Proximal Jejunum

In [ ]:
filtered_cells = df[df['unique_region']== 'B005_Proximal Jejunum']

filtered_probabilities_df = probabilities_df.loc[filtered_cells.index]

filtered_cells

Plotting for the paper to show zoomed in on the follicle areas.

In [ ]:
import matplotlib.pyplot as plt

# Step 1: Retrieve assigned neighborhoods and probabilities for the filtered cells
assigned_neighborhoods = filtered_cells['Neighborhood']

assigned_probabilities = filtered_probabilities_df.reindex(filtered_cells.index).apply(
    lambda row: row[filtered_cells.loc[row.name, 'Neighborhood']], axis=1
)

# Step 2: Create a new DataFrame with x, y, assigned neighborhood, probability, and region
visualization_df = pd.DataFrame({
    'x': filtered_cells['x'],
    'y': filtered_cells['y'],
    'assigned_neighborhood': assigned_neighborhoods,
    'assigned_probability': assigned_probabilities,
    'unique_region': filtered_cells['unique_region']
})

# Step 3: Filter for the desired unique_region
desired_region = 'B005_Proximal Jejunum'
filtered_region_df = visualization_df[visualization_df['unique_region'] == desired_region]

# Set zoom limits (update these values to zoom as needed)
xlim_coords = (0, 6000)  # Example: zoom in on x from 1000 to 2000
ylim_coords = (0, 8000)  # Example: zoom in on y from 1500 to 2500

# Step 4: Create the scatter plot without a colorbar
plt.figure(figsize=(10, 10), dpi=300)
plt.scatter(
    filtered_region_df['x'], filtered_region_df['y'],
    c=filtered_region_df['assigned_probability'], cmap='viridis', alpha=0.75, s=6
)

# Set zoom limits
plt.xlim(*xlim_coords)
plt.ylim(*ylim_coords)

# Add title
plt.title(f'Assigned Probability\n({desired_region})', fontsize=35)

# Hide axes
plt.axis('off')

# Adjust layout
plt.tight_layout()

# Show the plot
plt.show()
